# 1. Import Libraries

In [57]:
# Data Manipulation
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Classification Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from imblearn.metrics import geometric_mean_score
# Model Saving
import joblib

In [58]:
print("Libraries imported successfully.")

Libraries imported successfully.


## 2: Loading the Dataset

In [30]:
df = pd.read_csv("../data/Delhi Accident Data.csv")
# Display first 5 rows
df.head()

,YEAR,DISTRICT,VEHICLE AT FAULT,VICTIM,TYPE OF ACCIDENT,# INJURED,# KILLED,Unnamed: 7,Unnamed: 8,Unnamed: 9
0,2008,NORTH WEST DELHI,UNKNOWN,CAR,FATAL ACCIDENT,0,1,NaN,NaN,NaN
1,2008,NORTH WEST DELHI,UNKNOWN,CYCLE,SIMPLE ACCIDENT,1,0,NaN,NaN,NaN
2,2008,NEW DELHI,HTV/GDS,PEDESTRIAN,FATAL ACCIDENT,0,1,NaN,NaN,NaN
3,2008,EAST DELHI,S/C&M/C,PEDESTRIAN,SIMPLE ACCIDENT,1,0,NaN,NaN,NaN
4,2008,SHAHDARA,S/C&M/C,PEDESTRIAN,SIMPLE ACCIDENT,1,0,NaN,NaN,NaN


In [31]:
# Shape of the dataset
print("Rows and Columns:", df.shape)

Rows and Columns: (75748, 10)


In [32]:
# Column names
print(df.columns.tolist())

['YEAR', 'DISTRICT', 'VEHICLE AT FAULT', 'VICTIM', 'TYPE OF ACCIDENT', '# INJURED', '# KILLED ', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9']


In [33]:
# Basic information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 75748 entries, 0 to 75747
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   YEAR              75748 non-null  int64  
 1   DISTRICT          75748 non-null  str    
 2   VEHICLE AT FAULT  75748 non-null  str    
 3   VICTIM            75748 non-null  str    
 4   TYPE OF ACCIDENT  75748 non-null  str    
 5   # INJURED         75748 non-null  int64  
 6   # KILLED          75748 non-null  int64  
 7   Unnamed: 7        0 non-null      float64
 8   Unnamed: 8        0 non-null      float64
 9   Unnamed: 9        1 non-null      str    
dtypes: float64(2), int64(3), str(5)
memory usage: 5.8 MB


In [34]:
# Statistical summary
df.describe(include="all")

,YEAR,DISTRICT,VEHICLE AT FAULT,VICTIM,TYPE OF ACCIDENT,# INJURED,# KILLED,Unnamed: 7,Unnamed: 8,Unnamed: 9
count,75748.000000,75748,75748,75748,75748,75748.000000,75748.000000,0.0,0.0,1
unique,NaN,13,34,41,3,NaN,NaN,NaN,NaN,1
top,NaN,SOUTH EAST DELHI,PVT CAR,PEDESTRIAN,SIMPLE ACCIDENT,NaN,NaN,NaN,NaN,
freq,NaN,10096,22043,31872,55883,NaN,NaN,NaN,NaN,1
mean,2012.446863,NaN,NaN,NaN,NaN,0.955682,0.248640,NaN,NaN,NaN
std,2.868879,NaN,NaN,NaN,NaN,0.840229,0.450093,NaN,NaN,NaN
min,2008.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN
25%,2010.000000,NaN,NaN,NaN,NaN,1.000000,0.000000,NaN,NaN,NaN
50%,2013.000000,NaN,NaN,NaN,NaN,1.000000,0.000000,NaN,NaN,NaN
75%,2015.000000,NaN,NaN,NaN,NaN,1.000000,0.000000,NaN,NaN,NaN


## 3. Data Cleaning

Cleaned the dataset by:
- Removing unnecessary columns
- Removing extra spaces from column names
- Checking for missing values
- Checking for duplicate records

In [35]:
df.columns = df.columns.str.strip()

In [36]:
print(df.columns.tolist())

['YEAR', 'DISTRICT', 'VEHICLE AT FAULT', 'VICTIM', 'TYPE OF ACCIDENT', '# INJURED', '# KILLED', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9']


In [37]:
## Removing unnecessary columns
df.drop(columns=["Unnamed: 7", "Unnamed: 8", "Unnamed: 9"], inplace=True)

In [38]:
print(df.columns.tolist())

['YEAR', 'DISTRICT', 'VEHICLE AT FAULT', 'VICTIM', 'TYPE OF ACCIDENT', '# INJURED', '# KILLED']


In [39]:
df.isnull().sum()

YEAR                0
DISTRICT            0
VEHICLE AT FAULT    0
VICTIM              0
TYPE OF ACCIDENT    0
# INJURED           0
# KILLED            0
dtype: int64

In [40]:
## duplicates
duplicate_count = df.duplicated().sum()
print(f"Duplicate Rows: {duplicate_count}")

Duplicate Rows: 56819


In [41]:
print("Dataset Shape:", df.shape)

Dataset Shape: (75748, 7)


# 4. Feature Engineering

Preparing the dataset for classification by:

- Creating a new target variable (`TARGET`)
- Separating features and target
- Encoding categorical features into numerical values

In [42]:
# Create binary target
df["TARGET"] = (df["# INJURED"] > 0).astype(int)

# Display first few rows
df[["# INJURED", "TARGET"]].head(10)

,# INJURED,TARGET
0,0,0
1,1,1
2,0,0
3,1,1
4,1,1
5,1,1
6,2,1
7,1,1
8,1,1
9,1,1


In [43]:
df["TARGET"].value_counts()

TARGET
1    58938
0    16810
Name: count, dtype: int64

In [44]:
X = df.drop(columns=["# INJURED", "# KILLED", "TARGET"])
y = df["TARGET"]

In [45]:
label_encoders = {}

categorical_columns = X.select_dtypes(include="object").columns

for col in categorical_columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

In [46]:
X.head()

,YEAR,DISTRICT,VEHICLE AT FAULT,VICTIM,TYPE OF ACCIDENT
0,2008,5,33,6,0
1,2008,5,33,9,2
2,2008,2,17,25,0
3,2008,1,23,25,2
4,2008,7,23,25,2


In [47]:
X.head()

,YEAR,DISTRICT,VEHICLE AT FAULT,VICTIM,TYPE OF ACCIDENT
0,2008,5,33,6,0
1,2008,5,33,9,2
2,2008,2,17,25,0
3,2008,1,23,25,2
4,2008,7,23,25,2


In [48]:
joblib.dump(label_encoders, "../models/label_encoders_classifier.pkl")

['../models/label_encoders_classifier.pkl']

# 5. Train-Test Split

The dataset is divided into training and testing sets.

- Training set (80%) is used to train the model.
- Testing set (20%) is used to evaluate the model on unseen data.

The `stratify` parameter ensures that both sets maintain the same proportion of injury and no-injury cases.

In [49]:
# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [50]:
print("Training Features:", X_train.shape)
print("Testing Features :", X_test.shape)
print("Training Labels  :", y_train.shape)
print("Testing Labels   :", y_test.shape)

Training Features: (60598, 5)
Testing Features : (15150, 5)
Training Labels  : (60598,)
Testing Labels   : (15150,)


In [51]:
print("Training Class Distribution")
print(y_train.value_counts())

print("\nTesting Class Distribution")
print(y_test.value_counts())

Training Class Distribution
TARGET
1    47150
0    13448
Name: count, dtype: int64

Testing Class Distribution
TARGET
1    11788
0     3362
Name: count, dtype: int64


# 6. Decision Tree Classifier

A Decision Tree Classifier is trained to predict whether an accident will result in injuries.

The model is evaluated using:
- Accuracy
- F1 Score
- Confusion Matrix
- Classification Report

In [52]:
# Create the Decision Tree Classifier
dt_classifier = DecisionTreeClassifier(
    random_state=42,
    max_depth=5
)

# Train the model
dt_classifier.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at 

In [53]:
# Predict on test data
y_pred_dt = dt_classifier.predict(X_test)

In [54]:
# Accuracy
dt_accuracy = accuracy_score(y_test, y_pred_dt)

# F1 Score
dt_f1 = f1_score(y_test, y_pred_dt)

print(f"Accuracy : {dt_accuracy:.4f}")
print(f"F1 Score : {dt_f1:.4f}")

Accuracy : 0.9601
F1 Score : 0.9737


In [55]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_dt)

print(cm)

[[ 3362     0]
 [  605 11183]]


In [56]:
print(classification_report(y_test, y_pred_dt))

              precision    recall  f1-score   support

           0       0.85      1.00      0.92      3362
           1       1.00      0.95      0.97     11788

    accuracy                           0.96     15150
   macro avg       0.92      0.97      0.95     15150
weighted avg       0.97      0.96      0.96     15150



In [59]:
dt_gmean = geometric_mean_score(y_test, y_pred_dt)

print(f"Accuracy : {dt_accuracy:.4f}")
print(f"F1 Score : {dt_f1:.4f}")
print(f"G-Mean   : {dt_gmean:.4f}")

Accuracy : 0.9601
F1 Score : 0.9737
G-Mean   : 0.9740


## Decision Tree Evaluation

The Decision Tree Classifier achieved strong performance on the test dataset with an **Accuracy of 96.01%**, **F1 Score of 97.37%**, and **G-Mean of 97.40%**. The confusion matrix showed **0 false positives** and only **605 false negatives**, indicating that the model correctly classified the majority of accident records. Overall, the model demonstrated excellent predictive performance and serves as a strong baseline for comparison with AdaBoost and Stacking classifiers.